In [1]:
import pandas as pd
import numpy as np

In [2]:
nav = pd.read_csv("nav_history.csv")

transactions = pd.read_csv("investor_transactions.csv")

performance = pd.read_csv("scheme_performance.csv")

In [3]:
nav.head()

,amfi_code,date,nav
0,101,2026-01-01,16.00
1,103,2026-01-02,68.77
2,102,2026-01-03,186.75
3,105,2026-01-04,30.87
4,104,2026-01-05,17.63


In [4]:
# Convert date column
nav["date"] = pd.to_datetime(nav["date"])

# Sort
nav = nav.sort_values(["amfi_code", "date"])

# Forward fill missing NAV
nav["nav"] = nav.groupby("amfi_code")["nav"].ffill()

# Remove duplicate rows
nav = nav.drop_duplicates()

# Keep only NAV > 0
nav = nav[nav["nav"] > 0]

print(nav.head())

    amfi_code       date     nav
0         101 2026-01-01   16.00
5         101 2026-01-06   62.47
10        101 2026-01-11  192.11
17        101 2026-01-18  185.14
23        101 2026-01-24   21.00


In [5]:
# Convert transaction date
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])

# Standardize transaction types
transactions["transaction_type"] = (
    transactions["transaction_type"]
    .str.strip()
    .str.title()
)

transactions["transaction_type"] = transactions["transaction_type"].replace({
    "Sip":"SIP",
    "Lumpsum":"Lumpsum",
    "Redemption":"Redemption"
})

# Amount must be greater than zero
transactions = transactions[transactions["amount"] > 0]

# Keep only valid KYC values
valid_kyc = ["Verified","Pending","Rejected"]

transactions = transactions[
    transactions["kyc_status"].isin(valid_kyc)
]

print(transactions.head())

   transaction_id investor_id  amfi_code transaction_date transaction_type  \
0            1000     INV2000        102       2026-03-07          Lumpsum   
1            1001     INV2001        102       2026-03-22              SIP   
2            1002     INV2002        101       2026-02-19          Lumpsum   
3            1003     INV2003        103       2026-03-12              SIP   
4            1004     INV2004        105       2026-02-04       Redemption   

   amount state kyc_status  
0   12915    AP   Verified  
1   90192    TN   Rejected  
2   79104    TN   Rejected  
3   90166    AP   Rejected  
4   45587    AP    Pending  


In [6]:
# Convert return columns to numeric
performance["return_1y"] = pd.to_numeric(performance["return_1y"], errors="coerce")
performance["return_3y"] = pd.to_numeric(performance["return_3y"], errors="coerce")
performance["return_5y"] = pd.to_numeric(performance["return_5y"], errors="coerce")

# Remove rows with missing returns
performance = performance.dropna()

# Expense ratio validation
performance = performance[
    (performance["expense_ratio"] >= 0.1) &
    (performance["expense_ratio"] <= 2.5)
]

print(performance.head())

   amfi_code    scheme_name  return_1y  return_3y  return_5y  expense_ratio  \
0        101   Alpha Growth      -2.74      31.49      45.97           1.35   
1        105      Flexi Cap      11.58      17.89       9.76           0.97   
2        103  Balanced Plus       1.30      12.49      12.32           1.84   
3        105      Flexi Cap      17.48      32.77      17.94           0.54   
4        104    Income Fund      -4.26      27.10      21.95           2.15   

   aum_crore  
0       3124  
1        426  
2       3429  
3       1550  
4       2132  


In [7]:
print(nav.isnull().sum())

print(transactions.isnull().sum())

print(performance.isnull().sum())

amfi_code    0
date         0
nav          0
dtype: int64
transaction_id      0
investor_id         0
amfi_code           0
transaction_date    0
transaction_type    0
amount              0
state               0
kyc_status          0
dtype: int64
amfi_code        0
scheme_name      0
return_1y        0
return_3y        0
return_5y        0
expense_ratio    0
aum_crore        0
dtype: int64


In [8]:
print("NAV Duplicates:", nav.duplicated().sum())

print("Transaction Duplicates:", transactions.duplicated().sum())

print("Performance Duplicates:", performance.duplicated().sum())

NAV Duplicates: 0
Transaction Duplicates: 0
Performance Duplicates: 0


In [9]:
nav.to_csv("nav_history_cleaned.csv", index=False)

transactions.to_csv("investor_transactions_cleaned.csv", index=False)

performance.to_csv("scheme_performance_cleaned.csv", index=False)

print("All cleaned CSV files saved successfully.")

All cleaned CSV files saved successfully.


In [10]:
import sqlite3
import pandas as pd

In [11]:
conn = sqlite3.connect("bluestock_mf.db")


In [12]:
query = """
SELECT AVG(nav) AS Average_NAV
FROM fact_nav;
"""

pd.read_sql(query, conn)

,Average_NAV
0,None


In [13]:
query = """
SELECT SUM(amount) AS Total_Transaction_Amount
FROM fact_transactions;
"""

pd.read_sql(query, conn)

,Total_Transaction_Amount
0,None


In [14]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("bluestock_mf.db")

pd.read_sql("SELECT COUNT(*) AS Total_Rows FROM fact_nav;", conn)

,Total_Rows
0,0


In [15]:
import pandas as pd
from sqlalchemy import create_engine

In [16]:
nav = pd.read_csv("nav_history_cleaned.csv")

transactions = pd.read_csv("investor_transactions_cleaned.csv")

performance = pd.read_csv("scheme_performance_cleaned.csv")

In [17]:
engine = create_engine("sqlite:///bluestock_mf.db")

In [19]:
import pandas as pd

nav = pd.read_csv("nav_history_cleaned.csv")

print(nav.head())

print(nav.shape)

   amfi_code        date     nav
0        101  2026-01-01   16.00
1        101  2026-01-06   62.47
2        101  2026-01-11  192.11
3        101  2026-01-18  185.14
4        101  2026-01-24   21.00
(50, 3)


In [23]:
import sqlite3

conn = sqlite3.connect("bluestock_mf.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS fact_nav;")

conn.commit()
conn.close()

print("Old fact_nav table deleted.")

Old fact_nav table deleted.


In [24]:
import pandas as pd
from sqlalchemy import create_engine

nav = pd.read_csv("nav_history_cleaned.csv")

engine = create_engine("sqlite:///bluestock_mf.db")

nav.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

print("fact_nav table created successfully!")

fact_nav table created successfully!


In [25]:
conn = sqlite3.connect("bluestock_mf.db")

pd.read_sql("SELECT * FROM fact_nav LIMIT 5;", conn)

,amfi_code,date,nav
0,101,2026-01-01,16.00
1,101,2026-01-06,62.47
2,101,2026-01-11,192.11
3,101,2026-01-18,185.14
4,101,2026-01-24,21.00


In [26]:
query = """
SELECT AVG(nav) AS Average_NAV
FROM fact_nav;
"""

pd.read_sql(query, conn)

,Average_NAV
0,119.1032


In [27]:
transactions = pd.read_csv("investor_transactions_cleaned.csv")

transactions.to_sql(
    "fact_transactions",
    engine,
    if_exists="replace",
    index=False
)

50

In [28]:
performance = pd.read_csv("scheme_performance_cleaned.csv")

performance.to_sql(
    "fact_performance",
    engine,
    if_exists="replace",
    index=False
)

50

In [31]:
pd.read_sql("SELECT COUNT(*) FROM fact_transactions;", conn)



,COUNT(*)
0,50


In [30]:
pd.read_sql("SELECT COUNT(*) FROM fact_performance;", conn)

,COUNT(*)
0,50
